# Organization admin tasks

Runnable recipes for **organization administrators** using the official v2 **`istari_digital_client.Client`**. This notebook starts with the most common onboarding step after inviting a user: **grant tool access** so they can execute jobs.

It mirrors the **Admin → Users → Manage Tool Access** flow in the web app, but drives it from Python:

1. **List tools** (with functions) visible to your admin account.
2. **Find a user by email** in the organization.
3. **Grant executor access** to every tool.
4. **Verify** the user's tool permissions.

Uses [`samples/.env`](../.env) for `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN`. The token must belong to an **organization administrator** (same privileges as the Admin shield in the web app).

See also the [Onboard your organization](https://docs.istaridigital.com/tutorials/org-admin/onboard-your-organization) tutorial and [User Management — Manage Tool Access](https://docs.istaridigital.com/users/admin-guide/user-management#manage-tool-access-for-a-user).

### Prerequisites

- **`istari-digital-client`** and **`python-dotenv`** (from cookbook root: `uv sync --group dev`).
- Org-admin PAT in `samples/.env` (do not commit `.env`).
- A target user who is **Active** in **Admin → Users**.

### Install kernel (optional)

From the cookbook repository root:

```bash
uv sync --group dev
uv run python -m ipykernel install --user --name istari-client-cookbook --display-name "Python (istari-client-cookbook)"
```

Select **Python (istari-client-cookbook)** in the kernel picker.

### Running cells individually

Run **Setup → Connect**, then **Prep**, then §1–§4 in order.

## Setup

**Connect** loads credentials and constructs `Client`. **Prep** sets the target user's email and small helpers — edit before the demo cells.

### Connect

Load [`samples/.env`](../.env), build `Configuration`, and create `Client`.

In [ ]:
import os
from typing import Iterable, Iterator, List

import dotenv
from istari_digital_client import Configuration
from istari_digital_client.client import Client
from istari_digital_client.exceptions import ApiException
dotenv.load_dotenv("../.env")

registry_url = os.environ["ISTARI_REGISTRY_URL"]

client = Client(
    Configuration(
        registry_url=registry_url,
        registry_auth_token=os.environ["ISTARI_PERSONAL_ACCESS_TOKEN"],
    )
)

me = client.get_current_user()
print(f"Connected: {registry_url}")
print(f"Signed in as: {me.display_name} ({me.email})")

### Prep

Set `USER_EMAIL` to the user's **Email** column in **Admin → Users** (case-insensitive match). Helpers below paginate tools/functions and build access payloads.

In [ ]:
PAGE_SIZE = 100
from istari_digital_client.v2.models import User, Tool, ToolInclude


def find_user_by_email(users: Iterable[User], email: str) -> User:
    needle = email.casefold().strip()
    matches = [u for u in users if (u.email or "").casefold().strip() == needle]
    if not matches:
        raise LookupError(f"No user with email {email!r}")
    if len(matches) > 1:
        names = ", ".join(f"{u.display_name} <{u.email}>" for u in matches)
        raise LookupError(f"Multiple users match {email!r}: {names}")
    return matches[0]


def iter_tools(client: Client, *, page_size: int = PAGE_SIZE) -> Iterator[Tool]:
    page = 1
    while True:
        result = client.list_tools(
            page=page,
            size=page_size,
            include=[ToolInclude.FUNCTIONS],
        )
        items = result.items or []
        yield from items
        if not items or page >= (result.pages or page):
            break
        page += 1


def prompt_tools(tools: List[Tool]) -> None:
    print(f"Tools ({len(tools)}):")
    for tool in sorted(tools, key=lambda t: (t.name or "").casefold()):
        n = len(tool.functions or [])
        print(f"  {tool.name}: {n} function(s)")


def existing_execute_tool_ids(client: Client, user_id: str) -> set[str]:
    try:
        permissions = client.list_resource_type_permissions(
            subject_type=PermissionSubjectType.USER,
            subject_id=user_id,
            resource_type=PermissionResourceType.TOOL,
            permission=Permission.EXECUTE,
        )
        return {p.resource_id for p in permissions if p.resource_id}
    except ApiException:
        return set()

## §1 · List tools and functions

`list_tools` with `include=[ToolInclude.FUNCTIONS]` returns the tools visible in **Manage Tool Access**. Run this first to see which tools (and how many functions each has) will receive **executor** access.

In [ ]:
tools = list(iter_tools(client))

prompt_tools(tools)
print(f"\nTotal tools to grant: {len(tools)}")

## §2 · Find user by email

`list_users(user_state=UserStateOption.ACTIVE)` returns organization members. We match `USER_EMAIL` against each user's `email` (case-insensitive).

In [ ]:
from istari_digital_client.v2.models.user_state_option import UserStateOption

# Edit for your pilot user — matches the "Email" column in Admin → Users.
USER_EMAIL = "rderbier@istaridigital.com"

users = client.list_users(user_state=UserStateOption.ACTIVE)
target_user = find_user_by_email(users, USER_EMAIL)

print(f"Target user: {target_user.email}")
print(f"  name:  {target_user.display_name}")
print(f"  id:    {target_user.id}")
print(f"  state: {target_user.provider_user_state}")

## §3 · Grant executor access to all tools

**Manage Tool Access** has two levels — the checkboxes reflect which API resource gets the **`executor`** relation:

| UI checkbox | API resource | Effect |
|---|---|---|
| Tool row — **filled / black and checked** | `tool` | Executor on the **tool** — all current functions **and** any functions added later. |
| Function row — **checked only** (tool row not fully checked) | `function` | Executor on **selected functions only** — does **not** cover future functions. |

This notebook grants at the **tool** level (`PUT /api/v2/access/user/{user_id}/tool`), matching a fully checked tool row in the UI. One [`update_access_for_resource_type`](https://docs.istaridigital.com/developers/SDK/api_reference/05-access#update_access_for_resource_type) call is the same **Save** action as **Manage Tool Access**.

Re-run §1 first if your agent modules changed.

In [ ]:
from istari_digital_client.v2.models import (
    AccessRelation,
    AccessResourceType,
    AccessSubjectType,
    UpdateAccessRelationshipList,
    UpdateAccessRelationshipListItem,
    )

if not tools:
    raise RuntimeError("No tools found in §1 — connect agents/modules before granting tool access.")

confirm = input(
    f"Grant executor access to {len(tools)} tool(s) for {target_user.email}? [y/N] "
).strip().casefold()
if confirm not in {"y", "yes"}:
    print("Skipped — no access changes made.")
else:
    # Tool-level executor (filled tool checkbox in Manage Tool Access):
    # all current functions and any functions added to the tool later.
    # Function-level executor would be AccessResourceType.FUNCTION per function —
    # selected functions only, not future ones.
    body = UpdateAccessRelationshipList(
        resources=[
            UpdateAccessRelationshipListItem(
                relation=AccessRelation.EXECUTOR,
                resource_id=tool.id,
            )
            for tool in tools
        ]
    )
    client.update_access_for_resource_type(
        AccessSubjectType.USER,
        target_user.id,
        AccessResourceType.TOOL,
        body,
    )
    print(f"Granted executor access on {len(tools)} tool(s) for {target_user.email}.")

## §4 · Verify tool permissions

`list_resource_type_permissions` lists resources of a type where the user holds a given permission. After §3, the target user should have **`execute`** on every tool you granted.

In [ ]:
from istari_digital_client.v2.models import (
    Permission,
    PermissionResourceType,
    PermissionSubjectType,
)
executable_ids = existing_execute_tool_ids(client, target_user.id)
expected_ids = {tool.id for tool in tools}
missing = expected_ids - executable_ids

print(f"Execute permission on {len(executable_ids)} tool(s) for {target_user.email}")
if not executable_ids and tools:
    print("Could not list permissions — check Admin → Users → Manage Tool Access in the web app.")
elif missing:
    print(f"Warning: {len(missing)} tool(s) from §1 still lack execute permission.")
else:
    print("All tools from §1 are executable for this user.")
